# 03 - Orchestrator Pattern
> Claude como maestro coordenando subagentes especializados

**Modulo:** `04_agentes` | **Feito com:** [Claude](https://claude.ai) (Anthropic)

---


In [ ]:
import os
from dotenv import load_dotenv
import anthropic

load_dotenv()
client = anthropic.Anthropic(api_key=os.getenv('ANTHROPIC_API_KEY'))
HAIKU  = 'claude-haiku-4-5-20251001'
SONNET = 'claude-sonnet-4-5'

def ask(prompt, system=None, model=HAIKU, max_tokens=1024):
    kw = dict(model=model, max_tokens=max_tokens,
              messages=[{'role':'user','content':prompt}])
    if system: kw['system'] = system
    return client.messages.create(**kw).content[0].text

print('Pronto!')

In [ ]:
ESPECIALISTAS={
    'pesquisador':('pesquisa','Seja factual e estruturado.'),
    'redator':('redacao','Escreva de forma clara e profissional.'),
    'revisor':('revisao','Identifique erros e sugira melhorias.')
}

TOOLS=[{'name':'delegar','description':'Delega tarefa a subagente especializado.',
    'input_schema':{'type':'object','properties':{
        'agente':{'type':'string','enum':list(ESPECIALISTAS.keys())},
        'tarefa':{'type':'string'}},'required':['agente','tarefa']}}]

def delegar(agente,tarefa):
    papel,instrucao=ESPECIALISTAS[agente]
    print(f'  [{agente}]: {tarefa[:60]}...')
    return ask(tarefa,system=f'Voce e especialista em {papel}. {instrucao}',model=HAIKU)

def orquestrador(pedido):
    msgs=[{'role':'user','content':pedido}]
    sys_='Voce orquestra conteudo. Divida em subtarefas e delegue aos especialistas.'
    while True:
        r=client.messages.create(model=SONNET,max_tokens=2048,tools=TOOLS,system=sys_,messages=msgs)
        if r.stop_reason=='end_turn': return r.content[0].text
        msgs.append({'role':'assistant','content':r.content})
        res=[]
        for b in r.content:
            if b.type=='tool_use':
                res.append({'type':'tool_result','tool_use_id':b.id,'content':delegar(**b.input)})
        msgs.append({'role':'user','content':res})

print(orquestrador('Crie um post de blog sobre os beneficios do Python para iniciantes.'))

## Exercicios
> Complete os exercicios abaixo.


In [ ]:
# Seu codigo aqui


## Proximos passos
- Proximo notebook do modulo
- [docs.anthropic.com](https://docs.anthropic.com)
